In [1]:
import numpy as np
import torch
import qtorch
from qtorch.quant import bfloat16_boundedPosit8_quantize

Using /home/himeshi/.cache/torch_extensions/py310_cu126 as PyTorch extensions root...
Emitting ninja build file /home/himeshi/.cache/torch_extensions/py310_cu126/quant_cpu/build.ninja...
Building extension module quant_cpu...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/2] c++ -MMD -MF quant_cpu.o.d -DTORCH_EXTENSION_NAME=quant_cpu -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1016\" -isystem /home/himeshi/conga25/conga25env/lib/python3.10/site-packages/torch/include -isystem /home/himeshi/conga25/conga25env/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/himeshi/.pyenv/versions/3.10.4/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=1 -fPIC -std=c++17 -O3 -std=c++17 -fPIC -c /home/himeshi/conga25/QPyTorch/qtorch/quant/quant_cpu/quant_cpu.cpp -o quant_cpu.o 
[2/2] c++ quant_cpu.o bit_helper.o sim_helper.o -shared -shared -L/home/himeshi/conga25/conga25env/lib/python3.10/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o quant_cpu.so


Loading extension module quant_cpu...
Using /home/himeshi/.cache/torch_extensions/py310_cu126 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/himeshi/.cache/torch_extensions/py310_cu126/quant_cuda/build.ninja...
/home/himeshi/conga25/conga25env/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2356: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module quant_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.


Loading extension module quant_cuda...


Testing posit with maximum regime. 20 in 8,1,3 should be 0|111|0|010.

In [2]:
test_tensor = torch.tensor(20.0)
test_tensor_bf16 = test_tensor.to(torch.bfloat16)
bp = bfloat16_boundedPosit8_quantize(test_tensor_bf16, nsize=8, es=1, rs = 3)
print(bp)

p: 0 32512 0
temp_p: 114
1 temp_p: 114
2 temp_p: 29184
3 bf: 1101004800 1114636288 1016070144 
4 p: 29184
5 p: 29184
temp: 114
20 01110010 temp:114
1 regime: 4, regime_length: 3
2 regime: -4, regime_length: 3
bf: 32
posit:16800
tensor(20., dtype=torch.bfloat16)


Posit larger than maximum. 65 should return MAXREALINT. i.e. 0|111|1|111.

In [3]:
test_tensor = torch.tensor(65.0)
test_tensor_bf16 = test_tensor.to(torch.bfloat16)
bp = bfloat16_boundedPosit8_quantize(test_tensor_bf16, nsize=8, es=1, rs = 3)
print(bp)

p: 32512 32512 -1
temp_p: 120
1 temp_p: 120
2 temp_p: 30720
3 bf: 1115815936 1114636288 1016070144 
4 p: 32512
5 p: 32512
temp: 127
65 01111111 temp:127
1 regime: 4, regime_length: 3
2 regime: -4, regime_length: 3
bf: 240
posit:17008
tensor(60., dtype=torch.bfloat16)


Testing another posit with maximum regime + rounding.

In [4]:
test_tensor = torch.tensor(33.0)
test_tensor_bf16 = test_tensor.to(torch.bfloat16)
bp = bfloat16_boundedPosit8_quantize(test_tensor_bf16, nsize=8, es=1, rs = 3)
print(bp)

p: 0 32512 0
temp_p: 120
1 temp_p: 120
2 temp_p: 30720
3 bf: 1107558400 1114636288 1016070144 
4 p: 30720
5 p: 30720
temp: 120
33 01111000 temp:120
1 regime: 4, regime_length: 3
2 regime: -4, regime_length: 3
bf: 128
posit:16896
tensor(32., dtype=torch.bfloat16)


Testing MINREALINT saturation rounding.

In [5]:
test_tensor = torch.tensor(0.0001)
test_tensor_bf16 = test_tensor.to(torch.bfloat16)
bp = bfloat16_boundedPosit8_quantize(test_tensor_bf16, nsize=8, es=1, rs=3)
print(bp)

p: 256 32512 0
temp_p: 0
1 temp_p: 1
2 temp_p: 256
3 bf: 953286656 1114636288 1016070144 
4 p: 256
5 p: 256
temp: 1
0.000100136 00000001 temp:1
1 regime: 6, regime_length: 3
2 regime: 6, regime_length: 3
bf: 16
posit:15504
tensor(0.0176, dtype=torch.bfloat16)


Testing maximum minimum regime.

In [7]:
test_tensor = torch.tensor(0.02)
test_tensor_bf16 = test_tensor.to(torch.bfloat16)
bp = bfloat16_boundedPosit8_quantize(test_tensor_bf16, nsize=8, es=1, rs=3)
print(bp)

p: 0 32512 0
temp_p: 9
1 temp_p: 9
2 temp_p: 2304
3 bf: 1017380864 1114636288 1016070144 
4 p: 2304
5 p: 2304
temp: 9
0.0200195 00001001 temp:9
1 regime: 6, regime_length: 3
2 regime: 6, regime_length: 3
bf: 144
posit:15632
tensor(0.0352, dtype=torch.bfloat16)
